In [1]:
# ============================================================
# KAGGLE ONE-CELL
# KS MULTI-FIDELITY QUERY-T TRAJECTORY DATASET GENERATOR
#
# FORMAL NEW DATASET:
#
#   LF:
#       second-order centered finite difference
#       N = 72
#       ETDRK4
#
#   HF:
#       3/2-rule dealiased Fourier pseudospectral
#       N = 256
#       ETDRK4
#
# ============================================================
#
# PDE
# ---
#
#   u_t + u u_x + u_xx + beta u_xxxx = 0,
#   x in [0, 2*pi), periodic.
#
#
# ============================================================
# FORMAL DATASET SETTING
# ============================================================
#
# beta:
#       200 uniformly spaced values in [0.070, 0.085]
#
# trajectories/operator:
#       20 independent GRF initial conditions
#
# formal cross-operator split:
#       Group A = 100 beta operators
#       Group B = 100 beta operators
#       fixed seed = 42
#
# LF:
#       N = 72
#       second-order centered FD
#       ETDRK4
#
# HF:
#       N = 256
#       Fourier pseudospectral
#       3/2-rule dealiasing
#       ETDRK4
#
# common learning grid:
#       Nr = 100
#
# time:
#       T_max = 3.25
#       solver dt = 0.001
#
# saved BASE snapshots (100):
#       2.505, 2.510, ..., 3.000   (dt_save=0.005)
#
# saved near-future OOD snapshots (50):
#       3.005, 3.010, ..., 3.250   (dt_save=0.005)
#
# All snapshots are exact fixed-step ETDRK4 states; no temporal interpolation.
# Natural MIXED Group-A pool = 100 BASE + 50 OOD, hence OOD fraction = 1/3.
#
#
# ============================================================
# INITIAL CONDITIONS
# ============================================================
#
#   u0(x) = f(sin^2(x/2))
#
# where f is a zero-mean squared-exponential Gaussian random field:
#
#   Cov[f(z),f(z')]
#
#       =
#
#   sigma^2 exp(-(z-z')^2/(2 ell^2))
#
# with:
#
#   ell   = 0.5
#   sigma = 1.0
#
#
# IMPORTANT:
#
# We first evaluate the IC on the canonical HF256 grid.
#
# Then:
#
#   canonical HF256 IC
#       -> LF72 by periodic Fourier resampling
#       -> common100 by periodic Fourier resampling
#
# Therefore LF and HF correspond to exactly the same underlying
# numerical initial state.
#
#
# ============================================================
# LF SPATIAL DISCRETIZATION
# ============================================================
#
# Conservative nonlinear flux:
#
#   f(u) = u^2/2
#
#   -f_x(x_i)
#
#       ~
#
#   -[f(u_{i+1}) - f(u_{i-1})]/(2 dx)
#
#
# Linear second derivative:
#
#   D2 u_i
#
#       =
#
#   (u_{i+1} - 2u_i + u_{i-1})/dx^2
#
#
# Fourth derivative:
#
#   D4 = D2(D2)
#
#
# Linear KS operator:
#
#   -u_xx - beta u_xxxx
#
#
# Nominal LF spatial accuracy:
#
#   O(dx^2)
#
#
# ============================================================
# HF SPATIAL DISCRETIZATION
# ============================================================
#
# Periodic Fourier pseudospectral
#
# Linear symbol:
#
#   L_k = k^2 - beta k^4
#
# Quadratic nonlinear term:
#
#   -u u_x
#
#       =
#
#   -0.5 d_x(u^2)
#
# evaluated using standard 3/2-rule zero padding.
#
#
# ============================================================
# MAIN SAVED ARRAYS
# ============================================================
#
# beta                  [200]
# times                 [150]
#
# x_lf                  [72]
# x_hf                  [256]
# x                     [100]
#
# u0_lf                 [200,20,72]
# u0_hf                 [200,20,256]
# u0                    [200,20,100]
#
# uc_raw                [200,20,150,72]
# uf_raw                [200,20,150,256]
#
# uc                    [200,20,150,100]
# uf                    [200,20,150,100]
#
# baseline_relL2        [200,20,150]
# baseline_corr         [200,20,150]
#
# lf_mean_drift         [200,20,150]
# hf_mean_drift         [200,20,150]
#
# operator_traj_seed    [200]
# group_a_op_idx        [100]
# group_b_op_idx        [100]
# train_op_idx          [100]  # legacy diagnostic alias = Group B
# test_op_idx           [100]  # legacy diagnostic alias = Group A
#
#
# ============================================================
# RESTART SAFETY
# ============================================================
#
# Output and per-operator cache are stored under /kaggle/working.
# Each beta/operator is cached independently.
#
# If Kaggle stops:
#
#   rerun the exact same cell
#
# and completed operators will be reused.
#
# ============================================================


from __future__ import annotations


import hashlib
import json
import math
import time

from dataclasses import dataclass, asdict
from pathlib import Path


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import resample

import torch


# ============================================================
# KAGGLE OUTPUT ROOT
# ============================================================

DATASET_OUTPUT_ROOT = Path("/kaggle/working")
DATASET_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 0. USER SETTINGS
# ============================================================

QUICK_TEST = False


DATASET_STEM = (
    "ks_mf_queryt_beta070_085_"
    "FD2LF72_SpecHF256_T3p25_"
    "B100late_OOD50_dt001_Nop200_Ntraj20_Nt150_Nr100"
)


OUTPUT_ROOT = (
    DATASET_OUTPUT_ROOT
    /
    DATASET_STEM
)


CACHE_DIR = (
    OUTPUT_ROOT
    /
    "operator_cache"
)


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


OVERWRITE_OPERATOR_CACHE = False

SAVE_COMPRESSED = True


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


REAL_DTYPE = torch.float64

COMPLEX_DTYPE = torch.complex128


LF_SOLVER_TAG = (
    "KS_FD2_CENTERED_ETDRK4_N72_V1"
)


HF_SOLVER_TAG = (
    "KS_SPECTRAL_3OVER2_ETDRK4_N256_V1"
)


# ============================================================
# 1. CONFIG
# ============================================================

@dataclass(frozen=True)
class Config:

    # --------------------------------------------------------
    # Domain
    # --------------------------------------------------------

    L: float = (
        2.0
        *
        math.pi
    )


    # --------------------------------------------------------
    # Operator family
    # --------------------------------------------------------

    beta_min: float = 0.070

    beta_max: float = 0.085


    # --------------------------------------------------------
    # Dataset size
    # --------------------------------------------------------

    n_op: int = 200

    n_traj: int = 20

    n_train_op: int = 100

    n_test_op: int = 100


    # --------------------------------------------------------
    # Fidelity
    # --------------------------------------------------------

    nx_lf: int = 72

    nx_hf: int = 256

    nr: int = 100


    # --------------------------------------------------------
    # Time
    # --------------------------------------------------------

    t_final: float = 3.25

    # Fixed ETDRK4 step. All saved times are exact multiples of dt.
    dt: float = 0.001


    # --------------------------------------------------------
    # BASE / near-future OOD saved-time pools
    # --------------------------------------------------------

    base_t_min: float = 2.505
    base_t_max: float = 3.0
    base_snapshot_dt: float = 0.005
    n_base_time: int = 100

    ood_t_min: float = 3.005
    ood_t_max: float = 3.25
    ood_snapshot_dt: float = 0.005
    n_ood_time: int = 50


    # --------------------------------------------------------
    # ETDRK4 contour quadrature
    # --------------------------------------------------------

    etdrk4_m: int = 32


    # --------------------------------------------------------
    # GRF
    # --------------------------------------------------------

    grf_lengthscale: float = 0.5

    grf_sigma: float = 1.0

    grf_z_grid_n: int = 256

    grf_jitter: float = 1.0e-10


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    ic_base_seed: int = 20260924

    operator_seed_stride: int = 1000003

    split_seed: int = 42


CFG = Config()


# ============================================================
# 2. QUICK / FULL DATASET SIZE
# ============================================================

if QUICK_TEST:

    N_OP = 6

    N_TRAJ = 4

    N_TRAIN = 3

    N_TEST = 3


    dataset_stem = (
        "QUICK_"
        "ks_mf_queryt_beta070_085_"
        "FD2LF72_SpecHF256_T3p25_"
        "B100late_OOD50_dt001_Nop6_Ntraj4_Nt150_Nr100"
    )


    OUTPUT_ROOT = (
        DATASET_OUTPUT_ROOT
        /
        dataset_stem
    )


    CACHE_DIR = (
        OUTPUT_ROOT
        /
        "operator_cache"
    )


    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    CACHE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


else:

    N_OP = CFG.n_op

    N_TRAJ = CFG.n_traj

    N_TRAIN = CFG.n_train_op

    N_TEST = CFG.n_test_op

    dataset_stem = DATASET_STEM


assert (
    N_TRAIN
    +
    N_TEST
    ==
    N_OP
)


assert (
    CFG.nx_lf
    %
    2
    ==
    0
)


assert (
    CFG.nx_hf
    %
    2
    ==
    0
)


# ============================================================
# 3. TIME GRID
# ============================================================

# The integrator starts from t=0, but t=0 is not a saved/training token.
# Save only the explicitly requested 100 BASE + 50 OOD solver states.

n_steps = int(round(CFG.t_final / CFG.dt))
assert abs(n_steps * CFG.dt - CFG.t_final) < 1.0e-12

base_times = CFG.base_t_min + np.arange(CFG.n_base_time, dtype=np.float64) * CFG.base_snapshot_dt
ood_times = CFG.ood_t_min + np.arange(CFG.n_ood_time, dtype=np.float64) * CFG.ood_snapshot_dt

assert base_times.shape == (100,)
assert ood_times.shape == (50,)
assert abs(base_times[0] - 2.505) < 1.0e-12
assert abs(base_times[-1] - 3.00) < 1.0e-12
assert abs(ood_times[0] - 3.005) < 1.0e-12
assert abs(ood_times[-1] - 3.250) < 1.0e-12
assert base_times[-1] < ood_times[0]
assert abs((ood_times[0] - base_times[-1]) - 0.005) < 1.0e-12
assert np.allclose(np.diff(base_times), 0.005, rtol=0.0, atol=1.0e-12)
assert np.allclose(np.diff(ood_times), 0.005, rtol=0.0, atol=1.0e-12)

times = np.concatenate([base_times, ood_times]).astype(np.float64)
N_TIME = int(times.size)
assert N_TIME == 150
assert np.all(np.diff(times) > 0.0)
assert np.allclose(np.diff(times), 0.005, rtol=0.0, atol=1.0e-12)

snapshot_steps = np.rint(times / CFG.dt).astype(np.int64)
assert np.all(snapshot_steps >= 1)
assert np.all(snapshot_steps <= n_steps)
assert np.unique(snapshot_steps).size == N_TIME
assert np.allclose(snapshot_steps.astype(np.float64) * CFG.dt, times, rtol=0.0, atol=1.0e-12)

snapshot_step_to_index = np.full(n_steps + 1, -1, dtype=np.int64)
snapshot_step_to_index[snapshot_steps] = np.arange(N_TIME, dtype=np.int64)

base_time_idx = np.arange(0, CFG.n_base_time, dtype=np.int64)
ood_time_idx = np.arange(CFG.n_base_time, N_TIME, dtype=np.int64)
mixed_time_idx = np.arange(0, N_TIME, dtype=np.int64)

base_time_mask = np.zeros(N_TIME, dtype=bool)
base_time_mask[base_time_idx] = True
ood_time_mask = np.zeros(N_TIME, dtype=bool)
ood_time_mask[ood_time_idx] = True
mixed_time_mask = np.ones(N_TIME, dtype=bool)
mixed_times = times.copy()

assert base_time_idx.shape == (100,)
assert ood_time_idx.shape == (50,)
assert mixed_time_idx.shape == (150,)

NATURAL_GROUP_A_OOD_FRACTION = len(ood_time_idx) / len(mixed_time_idx)
assert abs(NATURAL_GROUP_A_OOD_FRACTION - 1.0 / 3.0) < 1.0e-12


# ============================================================
# 4. PRINT CONFIG
# ============================================================

print("=" * 130)

print(
    "KS MULTI-FIDELITY QUERY-T DATASET GENERATOR"
)

print("=" * 130)


print(
    "DEVICE              :",
    DEVICE,
)


print(
    "QUICK_TEST          :",
    QUICK_TEST,
)


print(
    "OUTPUT_ROOT         :",
    OUTPUT_ROOT,
)


print(
    "PDE                 : "
    "u_t + u*u_x + u_xx + beta*u_xxxx = 0"
)


print(
    f"beta range          : "
    f"[{CFG.beta_min:.6f}, {CFG.beta_max:.6f}]"
)


print(
    "N operators         :",
    N_OP,
)


print(
    "trajectories/op     :",
    N_TRAJ,
)


print(
    "train / test beta   :",
    N_TRAIN,
    "/",
    N_TEST,
)


print(
    "LF / HF / Nr        :",
    CFG.nx_lf,
    "/",
    CFG.nx_hf,
    "/",
    CFG.nr,
)


print(
    "LF solver           : "
    "2nd-order centered FD + ETDRK4"
)


print(
    "HF solver           : "
    "3/2-dealiased Fourier pseudospectral + ETDRK4"
)


print(
    "T_max / solver dt   :",
    CFG.t_final,
    "/",
    CFG.dt,
)


print(
    "BASE snapshot dt    :",
    CFG.base_snapshot_dt,
)

print(
    "OOD snapshot dt     :",
    CFG.ood_snapshot_dt,
)


print(
    "N snapshots         :",
    N_TIME,
)


print(
    "saved times         :",
    f"{times[0]:.3f} ... {times[-1]:.3f}",
)


print(
    "base pool           :",
    (
        f"{base_times[0]:.3f} ... "
        f"{base_times[-1]:.3f} "
        f"({len(base_time_idx)} slots)"
    ),
)


print(
    "near-future OOD    :",
    (
        f"{ood_times[0]:.3f} ... "
        f"{ood_times[-1]:.3f} "
        f"({len(ood_time_idx)} slots)"
    ),
)


print(
    "mixed A pool       :",
    (
        f"{mixed_times[0]:.3f} ... "
        f"{mixed_times[-1]:.3f} "
        f"({len(mixed_time_idx)} slots)"
    ),
)


print(
    "IC                  : "
    "u0(x)=f(sin^2(x/2))"
)


print(
    "GRF                 : "
    f"SE kernel, ell={CFG.grf_lengthscale}, "
    f"sigma={CFG.grf_sigma}"
)


print(
    "fresh IC base seed  :",
    CFG.ic_base_seed,
)


print("=" * 130)


# ============================================================
# 5. FIXED GRIDS / OPERATOR GRID / SPLIT
# ============================================================

x_lf = np.linspace(
    0.0,
    CFG.L,
    CFG.nx_lf,
    endpoint=False,
    dtype=np.float64,
)


x_hf = np.linspace(
    0.0,
    CFG.L,
    CFG.nx_hf,
    endpoint=False,
    dtype=np.float64,
)


x_common = np.linspace(
    0.0,
    CFG.L,
    CFG.nr,
    endpoint=False,
    dtype=np.float64,
)


beta = np.linspace(
    CFG.beta_min,
    CFG.beta_max,
    N_OP,
    dtype=np.float64,
)


rng_split = np.random.default_rng(
    CFG.split_seed
)


perm = rng_split.permutation(
    N_OP
)


train_op_idx = np.sort(
    perm[
        :N_TRAIN
    ]
).astype(
    np.int64
)


test_op_idx = np.sort(
    perm[
        N_TRAIN:
    ]
).astype(
    np.int64
)

# Formal cross-operator split used by NO-MIXED / MIXED training.
# Group B = first 100 entries of the seed-42 permutation; Group A = remainder.
group_b_op_idx = train_op_idx.copy()
group_a_op_idx = test_op_idx.copy()

if not QUICK_TEST:
    assert group_a_op_idx.shape == (100,)
    assert group_b_op_idx.shape == (100,)
    assert np.intersect1d(group_a_op_idx, group_b_op_idx).size == 0
    assert np.array_equal(
        np.sort(np.concatenate([group_a_op_idx, group_b_op_idx])),
        np.arange(200, dtype=np.int64),
    )


operator_traj_seed = np.asarray(

    [
        (
            CFG.ic_base_seed
            +
            i
            *
            CFG.operator_seed_stride
        )

        for i in range(
            N_OP
        )
    ],

    dtype=np.int64,
)


print(
    "\nOperator split:"
)


print(
    "  Group B:",
    group_b_op_idx,
)

print(
    "  Group A:",
    group_a_op_idx,
)


# ============================================================
# 6. GRF SAMPLER
# ============================================================

z_grf = np.linspace(
    0.0,
    1.0,
    CFG.grf_z_grid_n,
    endpoint=True,
    dtype=np.float64,
)


dz = (
    z_grf[
        :,
        None
    ]
    -
    z_grf[
        None,
        :
    ]
)


K_grf = (

    CFG.grf_sigma ** 2

    *

    np.exp(
        -0.5
        *
        (
            dz
            /
            CFG.grf_lengthscale
        ) ** 2
    )
)


K_grf += (
    CFG.grf_jitter
    *
    np.eye(
        CFG.grf_z_grid_n,
        dtype=np.float64,
    )
)


GRF_CHOL = np.linalg.cholesky(
    K_grf
)


print(
    "\nGRF:"
)


print(
    "  z grid n    :",
    CFG.grf_z_grid_n,
)


print(
    "  lengthscale :",
    CFG.grf_lengthscale,
)


print(
    "  sigma       :",
    CFG.grf_sigma,
)


print(
    "  jitter      :",
    CFG.grf_jitter,
)


def sample_f_bank(
    n_traj: int,
    seed: int,
) -> np.ndarray:

    """
    Draw independent samples from the zero-mean GP f(z).
    """

    rng = np.random.default_rng(
        int(
            seed
        )
    )


    xi = rng.standard_normal(
        (
            n_traj,
            CFG.grf_z_grid_n,
        )
    ).astype(
        np.float64
    )


    return (
        xi
        @
        GRF_CHOL.T
    )


def eval_f_bank_at_x(
    f_bank: np.ndarray,
    x: np.ndarray,
) -> np.ndarray:

    """
    u0(x) = f(sin^2(x/2))

    f_bank:
        [B,Nz]

    x:
        [Nx]

    return:
        [B,Nx]
    """


    z_query = (
        np.sin(
            0.5
            *
            x
        ) ** 2
    )


    out = np.empty(
        (
            f_bank.shape[
                0
            ],
            len(
                x
            ),
        ),
        dtype=np.float64,
    )


    for i in range(
        f_bank.shape[
            0
        ]
    ):

        out[
            i
        ] = np.interp(
            z_query,
            z_grf,
            f_bank[
                i
            ],
        )


    return out


# ============================================================
# 7. FOURIER RESAMPLING / METRICS
# ============================================================

EPS = 1.0e-12


def fourier_resample_real(
    arr: np.ndarray,
    n_out: int,
) -> np.ndarray:

    """
    Periodic Fourier resampling along last axis.
    """


    arr = np.asarray(
        arr,
        dtype=np.float64,
    )


    out = resample(
        arr,
        int(
            n_out
        ),
        axis=-1,
    )


    return np.asarray(
        out.real,
        dtype=np.float64,
    )


def canonicalize_hf_initial_state(
    u: np.ndarray,
) -> np.ndarray:

    """
    Apply the same even-grid Nyquist convention used by the
    HF pseudospectral solver.

    This produces the canonical HF numerical initial state.
    """


    u = np.asarray(
        u,
        dtype=np.float64,
    )


    n = u.shape[
        -1
    ]


    if (
        n
        %
        2
        !=
        0
    ):

        raise ValueError(
            "Expected even HF resolution."
        )


    v = np.fft.fft(
        u,
        axis=-1,
    )


    v[
        ...,
        n // 2,
    ] = (
        0.0
        +
        0.0j
    )


    return np.fft.ifft(
        v,
        axis=-1,
    ).real.astype(
        np.float64
    )


def row_rel_l2(
    pred: np.ndarray,
    true: np.ndarray,
) -> np.ndarray:


    pred = np.asarray(
        pred,
        dtype=np.float64,
    )


    true = np.asarray(
        true,
        dtype=np.float64,
    )


    num = np.linalg.norm(
        pred
        -
        true,
        axis=-1,
    )


    den = np.linalg.norm(
        true,
        axis=-1,
    )


    return (
        num
        /
        np.maximum(
            den,
            EPS,
        )
    )


def row_corr(
    a: np.ndarray,
    b: np.ndarray,
) -> np.ndarray:


    a = np.asarray(
        a,
        dtype=np.float64,
    )


    b = np.asarray(
        b,
        dtype=np.float64,
    )


    ac = (
        a
        -
        a.mean(
            axis=-1,
            keepdims=True,
        )
    )


    bc = (
        b
        -
        b.mean(
            axis=-1,
            keepdims=True,
        )
    )


    num = np.sum(
        ac
        *
        bc,
        axis=-1,
    )


    den = (
        np.linalg.norm(
            ac,
            axis=-1,
        )
        *
        np.linalg.norm(
            bc,
            axis=-1,
        )
    )


    return (
        num
        /
        np.maximum(
            den,
            EPS,
        )
    )


def describe_percent(
    x: np.ndarray,
) -> dict:


    x = (
        100.0
        *
        np.asarray(
            x,
            dtype=np.float64,
        ).reshape(
            -1
        )
    )


    return {

        "mean":
            float(
                np.mean(
                    x
                )
            ),

        "std":
            float(
                np.std(
                    x
                )
            ),

        "median":
            float(
                np.median(
                    x
                )
            ),

        "p90":
            float(
                np.percentile(
                    x,
                    90,
                )
            ),

        "p95":
            float(
                np.percentile(
                    x,
                    95,
                )
            ),

        "p99":
            float(
                np.percentile(
                    x,
                    99,
                )
            ),

        "min":
            float(
                np.min(
                    x
                )
            ),

        "max":
            float(
                np.max(
                    x
                )
            ),
    }


# ============================================================
# 8. LF FD2 + ETDRK4 COEFFICIENTS
# ============================================================

def make_lf_fd2_etdrk4_coefficients(
    n: int,
    beta_value: float,
) -> dict:


    dx = (
        CFG.L
        /
        n
    )


    freq = np.fft.fftfreq(
        n
    )


    theta = (
        2.0
        *
        np.pi
        *
        freq
    )


    # --------------------------------------------------------
    # Second-order centered D2 symbol
    # --------------------------------------------------------

    D2_np = (
        -4.0
        *
        np.sin(
            theta
            /
            2.0
        ) ** 2
        /
        dx ** 2
    )


    # --------------------------------------------------------
    # D4 = D2(D2)
    # --------------------------------------------------------

    D4_np = (
        D2_np ** 2
    )


    # --------------------------------------------------------
    # KS linear operator:
    #
    #   -u_xx - beta u_xxxx
    # --------------------------------------------------------

    L_np = (
        -D2_np
        -
        float(
            beta_value
        )
        *
        D4_np
    )


    dt = CFG.dt


    E_np = np.exp(
        dt
        *
        L_np
    )


    E2_np = np.exp(
        0.5
        *
        dt
        *
        L_np
    )


    j = np.arange(
        1,
        CFG.etdrk4_m + 1,
        dtype=np.float64,
    )


    r = np.exp(
        1j
        *
        np.pi
        *
        (
            j
            -
            0.5
        )
        /
        CFG.etdrk4_m
    )


    LR = (
        dt
        *
        L_np[
            :,
            None
        ]
        +
        r[
            None,
            :
        ]
    )


    Q_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    np.exp(
                        LR
                        /
                        2.0
                    )
                    -
                    1.0
                )
                /
                LR,
                axis=1,
            )
        )
    )


    f1_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    -4.0
                    -
                    LR
                    +
                    np.exp(
                        LR
                    )
                    *
                    (
                        4.0
                        -
                        3.0
                        *
                        LR
                        +
                        LR ** 2
                    )
                )
                /
                (
                    LR ** 3
                ),
                axis=1,
            )
        )
    )


    f2_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    2.0
                    +
                    LR
                    +
                    np.exp(
                        LR
                    )
                    *
                    (
                        -2.0
                        +
                        LR
                    )
                )
                /
                (
                    LR ** 3
                ),
                axis=1,
            )
        )
    )


    f3_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    -4.0
                    -
                    3.0
                    *
                    LR
                    -
                    LR ** 2
                    +
                    np.exp(
                        LR
                    )
                    *
                    (
                        4.0
                        -
                        LR
                    )
                )
                /
                (
                    LR ** 3
                ),
                axis=1,
            )
        )
    )


    def rt(
        x,
    ):

        return torch.from_numpy(
            np.asarray(
                x,
                dtype=np.float64,
            )
        ).to(
            DEVICE,
            dtype=REAL_DTYPE,
        )


    return {

        "dx":
            float(
                dx
            ),

        "E":
            rt(
                E_np
            )[
                None,
                :
            ],

        "E2":
            rt(
                E2_np
            )[
                None,
                :
            ],

        "Q":
            rt(
                Q_np
            )[
                None,
                :
            ],

        "f1":
            rt(
                f1_np
            )[
                None,
                :
            ],

        "f2":
            rt(
                f2_np
            )[
                None,
                :
            ],

        "f3":
            rt(
                f3_np
            )[
                None,
                :
            ],
    }


# ============================================================
# 9. LF SECOND-ORDER NONLINEAR TERM
# ============================================================

def nonlinear_lf_fd2_hat(
    vhat: torch.Tensor,
    dx: float,
) -> torch.Tensor:

    """
    Conservative centered approximation:

        f(u) = u^2/2

        -f_x
          ~
        -[f(u_{i+1}) - f(u_{i-1})]/(2 dx)
    """


    u = torch.fft.ifft(
        vhat,
        dim=-1,
    ).real


    flux = (
        0.5
        *
        u
        *
        u
    )


    flux_right = torch.roll(
        flux,
        shifts=-1,
        dims=-1,
    )


    flux_left = torch.roll(
        flux,
        shifts=1,
        dims=-1,
    )


    nonlinear = (
        -(
            flux_right
            -
            flux_left
        )
        /
        (
            2.0
            *
            dx
        )
    )


    return torch.fft.fft(
        nonlinear,
        dim=-1,
    ).to(
        COMPLEX_DTYPE
    )


# ============================================================
# 10. LF BATCH TRAJECTORY SOLVER
# ============================================================

@torch.no_grad()
def solve_lf_fd2_trajectory_batch(
    u0_native_np: np.ndarray,
    beta_value: float,
) -> np.ndarray:


    u0_native_np = np.asarray(
        u0_native_np,
        dtype=np.float64,
    )


    if u0_native_np.shape != (
        N_TRAJ,
        CFG.nx_lf,
    ):

        raise ValueError(
            "Unexpected LF initial-state shape: "
            f"{u0_native_np.shape}"
        )


    coeff = make_lf_fd2_etdrk4_coefficients(
        CFG.nx_lf,
        beta_value,
    )


    dx = coeff[
        "dx"
    ]


    E = coeff[
        "E"
    ].to(
        COMPLEX_DTYPE
    )


    E2 = coeff[
        "E2"
    ].to(
        COMPLEX_DTYPE
    )


    Q = coeff[
        "Q"
    ].to(
        COMPLEX_DTYPE
    )


    f1 = coeff[
        "f1"
    ].to(
        COMPLEX_DTYPE
    )


    f2 = coeff[
        "f2"
    ].to(
        COMPLEX_DTYPE
    )


    f3 = coeff[
        "f3"
    ].to(
        COMPLEX_DTYPE
    )


    u0_t = torch.from_numpy(
        u0_native_np
    ).to(
        DEVICE,
        dtype=REAL_DTYPE,
    )


    v = torch.fft.fft(
        u0_t,
        dim=-1,
    ).to(
        COMPLEX_DTYPE
    )


    trajectory_t = torch.empty(
        (
            N_TRAJ,
            N_TIME,
            CFG.nx_lf,
        ),
        dtype=REAL_DTYPE,
        device=DEVICE,
    )


    stored_snapshot_count = 0


    for step in range(
        1,
        n_steps + 1,
    ):

        Nv = nonlinear_lf_fd2_hat(
            v,
            dx,
        )


        a = (
            E2
            *
            v
            +
            Q
            *
            Nv
        )


        Na = nonlinear_lf_fd2_hat(
            a,
            dx,
        )


        b = (
            E2
            *
            v
            +
            Q
            *
            Na
        )


        Nb = nonlinear_lf_fd2_hat(
            b,
            dx,
        )


        c = (
            E2
            *
            a
            +
            Q
            *
            (
                2.0
                *
                Nb
                -
                Nv
            )
        )


        Nc = nonlinear_lf_fd2_hat(
            c,
            dx,
        )


        v = (
            E
            *
            v
            +
            f1
            *
            Nv
            +
            2.0
            *
            f2
            *
            (
                Na
                +
                Nb
            )
            +
            f3
            *
            Nc
        )


        # ----------------------------------------------------
        # Remove accumulated imaginary roundoff only.
        # --------------------------------------------------------

        if (
            step
            %
            100
            ==
            0
            or
            step
            ==
            n_steps
        ):

            u_real = torch.fft.ifft(
                v,
                dim=-1,
            ).real


            v = torch.fft.fft(
                u_real,
                dim=-1,
            ).to(
                COMPLEX_DTYPE
            )


        # ----------------------------------------------------
        # Snapshot at explicitly requested solver steps
        # ----------------------------------------------------

        save_idx = int(snapshot_step_to_index[step])
        if save_idx >= 0:
            trajectory_t[:, save_idx, :] = torch.fft.ifft(v, dim=-1).real
            stored_snapshot_count += 1


        # ----------------------------------------------------
        # Finite-state check
        # --------------------------------------------------------

        if (
            step
            %
            100
            ==
            0
            or
            step
            ==
            n_steps
        ):

            if not torch.isfinite(
                torch.view_as_real(
                    v
                )
            ).all():

                raise FloatingPointError(
                    "Non-finite LF state at "
                    f"beta={beta_value}, "
                    f"N={CFG.nx_lf}, "
                    f"step={step}."
                )


    if stored_snapshot_count != N_TIME:

        raise RuntimeError(
            f"LF expected {N_TIME} snapshots, "
            f"stored {stored_snapshot_count}."
        )


    return (
        trajectory_t
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float64
        )
    )


# ============================================================
# 11. HF FOURIER ETDRK4 COEFFICIENTS
# ============================================================

def make_hf_etdrk4_coefficients(
    n: int,
    beta_value: float,
) -> dict:

    """
    HF KS Fourier linear symbol:

        L_k = k^2 - beta k^4

    Domain:
        [0,2*pi)
    """


    k_np = np.fft.fftfreq(
        n,
        d=1.0
        /
        n,
    ).astype(
        np.float64
    )


    L_np = (
        k_np ** 2
        -
        float(
            beta_value
        )
        *
        k_np ** 4
    )


    dt = CFG.dt


    E_np = np.exp(
        dt
        *
        L_np
    )


    E2_np = np.exp(
        0.5
        *
        dt
        *
        L_np
    )


    j = np.arange(
        1,
        CFG.etdrk4_m + 1,
        dtype=np.float64,
    )


    r = np.exp(
        1j
        *
        np.pi
        *
        (
            j
            -
            0.5
        )
        /
        CFG.etdrk4_m
    )


    LR = (
        dt
        *
        L_np[
            :,
            None
        ]
        +
        r[
            None,
            :
        ]
    )


    Q_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    np.exp(
                        LR
                        /
                        2.0
                    )
                    -
                    1.0
                )
                /
                LR,
                axis=1,
            )
        )
    )


    f1_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    -4.0
                    -
                    LR
                    +
                    np.exp(
                        LR
                    )
                    *
                    (
                        4.0
                        -
                        3.0
                        *
                        LR
                        +
                        LR ** 2
                    )
                )
                /
                (
                    LR ** 3
                ),
                axis=1,
            )
        )
    )


    f2_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    2.0
                    +
                    LR
                    +
                    np.exp(
                        LR
                    )
                    *
                    (
                        -2.0
                        +
                        LR
                    )
                )
                /
                (
                    LR ** 3
                ),
                axis=1,
            )
        )
    )


    f3_np = (
        dt
        *
        np.real(
            np.mean(
                (
                    -4.0
                    -
                    3.0
                    *
                    LR
                    -
                    LR ** 2
                    +
                    np.exp(
                        LR
                    )
                    *
                    (
                        4.0
                        -
                        LR
                    )
                )
                /
                (
                    LR ** 3
                ),
                axis=1,
            )
        )
    )


    def rt(
        x,
    ):

        return torch.from_numpy(
            np.asarray(
                x,
                dtype=np.float64,
            )
        ).to(
            DEVICE,
            dtype=REAL_DTYPE,
        )


    return {

        "k":
            rt(
                k_np
            )[
                None,
                :
            ],

        "E":
            rt(
                E_np
            )[
                None,
                :
            ],

        "E2":
            rt(
                E2_np
            )[
                None,
                :
            ],

        "Q":
            rt(
                Q_np
            )[
                None,
                :
            ],

        "f1":
            rt(
                f1_np
            )[
                None,
                :
            ],

        "f2":
            rt(
                f2_np
            )[
                None,
                :
            ],

        "f3":
            rt(
                f3_np
            )[
                None,
                :
            ],
    }


# ============================================================
# 12. HF 3/2-RULE NONLINEAR TERM
# ============================================================

def nonlinear_hf_3over2(
    vhat: torch.Tensor,
    k: torch.Tensor,
) -> torch.Tensor:

    """
    Compute:

        N(u) = -u*u_x = -0.5*d_x(u^2)

    using 3/2-rule padded pseudospectral evaluation.
    """


    n = vhat.shape[
        -1
    ]


    if (
        n
        %
        2
        !=
        0
    ):

        raise ValueError(
            "HF native resolution must be even."
        )


    m = (
        3
        *
        n
    ) // 2


    if (
        m
        %
        2
        !=
        0
    ):

        raise ValueError(
            "3N/2 padded size must be even."
        )


    # --------------------------------------------------------
    # Native physical grid
    # --------------------------------------------------------

    u = torch.fft.ifft(
        vhat,
        dim=-1,
    ).real


    # --------------------------------------------------------
    # Native rFFT
    # --------------------------------------------------------

    ur = torch.fft.rfft(
        u,
        dim=-1,
    ).clone()


    # --------------------------------------------------------
    # Standard even-grid Nyquist convention
    # --------------------------------------------------------

    ur[
        ...,
        n // 2,
    ] = (
        0.0
        +
        0.0j
    )


    # --------------------------------------------------------
    # Pad N -> 3N/2
    # --------------------------------------------------------

    ur_pad = torch.zeros(

        u.shape[
            :-1
        ]
        +
        (
            m // 2 + 1,
        ),

        dtype=COMPLEX_DTYPE,

        device=DEVICE,
    )


    # DFT_N/N = DFT_M/M
    ur_pad[
        ...,
        : n // 2 + 1,
    ] = (
        ur
        *
        (
            m
            /
            n
        )
    )


    u_pad = torch.fft.irfft(
        ur_pad,
        n=m,
        dim=-1,
    )


    # --------------------------------------------------------
    # Square on padded grid
    # --------------------------------------------------------

    u2r_m = torch.fft.rfft(
        u_pad
        *
        u_pad,
        dim=-1,
    )


    # --------------------------------------------------------
    # Truncate back to native N
    # --------------------------------------------------------

    u2r_n = (
        u2r_m[
            ...,
            : n // 2 + 1,
        ]
        *
        (
            n
            /
            m
        )
    )


    # --------------------------------------------------------
    # Reconstruct full complex FFT
    # --------------------------------------------------------

    u2hat = torch.zeros_like(
        vhat
    )


    u2hat[
        ...,
        0,
    ] = u2r_n[
        ...,
        0,
    ]


    pos = u2r_n[
        ...,
        1 : n // 2,
    ]


    u2hat[
        ...,
        1 : n // 2,
    ] = pos


    u2hat[
        ...,
        n // 2 + 1 :,
    ] = torch.conj(
        torch.flip(
            pos,
            dims=[
                -1
            ],
        )
    )


    # Nyquist derivative is ambiguous on even grid.
    u2hat[
        ...,
        n // 2,
    ] = (
        0.0
        +
        0.0j
    )


    return (
        -0.5j
        *
        k.to(
            COMPLEX_DTYPE
        )
        *
        u2hat
    )


# ============================================================
# 13. HF TRAJECTORY SOLVER
# ============================================================

@torch.no_grad()
def solve_hf_spectral_trajectory_batch(
    u0_native_np: np.ndarray,
    beta_value: float,
) -> np.ndarray:


    u0_native_np = np.asarray(
        u0_native_np,
        dtype=np.float64,
    )


    if u0_native_np.shape != (
        N_TRAJ,
        CFG.nx_hf,
    ):

        raise ValueError(
            "Unexpected HF initial-state shape: "
            f"{u0_native_np.shape}"
        )


    coeff = make_hf_etdrk4_coefficients(
        CFG.nx_hf,
        beta_value,
    )


    k = coeff[
        "k"
    ]


    E = coeff[
        "E"
    ].to(
        COMPLEX_DTYPE
    )


    E2 = coeff[
        "E2"
    ].to(
        COMPLEX_DTYPE
    )


    Q = coeff[
        "Q"
    ].to(
        COMPLEX_DTYPE
    )


    f1 = coeff[
        "f1"
    ].to(
        COMPLEX_DTYPE
    )


    f2 = coeff[
        "f2"
    ].to(
        COMPLEX_DTYPE
    )


    f3 = coeff[
        "f3"
    ].to(
        COMPLEX_DTYPE
    )


    u0_t = torch.from_numpy(
        u0_native_np
    ).to(
        DEVICE,
        dtype=REAL_DTYPE,
    )


    v = torch.fft.fft(
        u0_t,
        dim=-1,
    ).to(
        COMPLEX_DTYPE
    )


    # Canonical even-grid convention.
    v[
        ...,
        CFG.nx_hf // 2,
    ] = (
        0.0
        +
        0.0j
    )


    trajectory_t = torch.empty(
        (
            N_TRAJ,
            N_TIME,
            CFG.nx_hf,
        ),
        dtype=REAL_DTYPE,
        device=DEVICE,
    )


    stored_snapshot_count = 0


    for step in range(
        1,
        n_steps + 1,
    ):

        Nv = nonlinear_hf_3over2(
            v,
            k,
        )


        a = (
            E2
            *
            v
            +
            Q
            *
            Nv
        )


        Na = nonlinear_hf_3over2(
            a,
            k,
        )


        b = (
            E2
            *
            v
            +
            Q
            *
            Na
        )


        Nb = nonlinear_hf_3over2(
            b,
            k,
        )


        c = (
            E2
            *
            a
            +
            Q
            *
            (
                2.0
                *
                Nb
                -
                Nv
            )
        )


        Nc = nonlinear_hf_3over2(
            c,
            k,
        )


        v = (
            E
            *
            v
            +
            f1
            *
            Nv
            +
            2.0
            *
            f2
            *
            (
                Na
                +
                Nb
            )
            +
            f3
            *
            Nc
        )


        # ----------------------------------------------------
        # Roundoff symmetry cleanup
        # --------------------------------------------------------

        if (
            step
            %
            100
            ==
            0
            or
            step
            ==
            n_steps
        ):

            u_real = torch.fft.ifft(
                v,
                dim=-1,
            ).real


            v = torch.fft.fft(
                u_real,
                dim=-1,
            ).to(
                COMPLEX_DTYPE
            )


            v[
                ...,
                CFG.nx_hf // 2,
            ] = (
                0.0
                +
                0.0j
            )


        # ----------------------------------------------------
        # Snapshot at explicitly requested solver steps
        # ----------------------------------------------------

        save_idx = int(snapshot_step_to_index[step])
        if save_idx >= 0:
            trajectory_t[:, save_idx, :] = torch.fft.ifft(v, dim=-1).real
            stored_snapshot_count += 1


        # ----------------------------------------------------
        # Finite check
        # --------------------------------------------------------

        if (
            step
            %
            100
            ==
            0
            or
            step
            ==
            n_steps
        ):

            if not torch.isfinite(
                torch.view_as_real(
                    v
                )
            ).all():

                raise FloatingPointError(
                    "Non-finite HF KS state at "
                    f"beta={beta_value}, "
                    f"N={CFG.nx_hf}, "
                    f"step={step}."
                )


    if stored_snapshot_count != N_TIME:

        raise RuntimeError(
            f"HF expected {N_TIME} snapshots, "
            f"stored {stored_snapshot_count}."
        )


    return (
        trajectory_t
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float64
        )
    )


# ============================================================
# 14. CACHE HELPERS
# ============================================================

def operator_cache_path(
    op_idx: int,
) -> Path:


    return (
        CACHE_DIR
        /
        f"operator_{op_idx:04d}.npz"
    )


def cache_is_valid(
    path: Path,
    op_idx: int,
    beta_value: float,
) -> bool:


    if not path.exists():

        return False


    try:

        with np.load(
            path,
            allow_pickle=False,
        ) as d:


            expected = {

                "lf_solver_tag",
                "hf_solver_tag",

                "op_idx",
                "beta",
                "traj_seed",

                "times",

                "u0_lf",
                "u0_hf",
                "u0",

                "uc_raw",
                "uf_raw",

                "uc",
                "uf",

                "baseline_relL2",
                "baseline_corr",

                "lf_mean_drift",
                "hf_mean_drift",
            }


            if not expected.issubset(
                set(
                    d.files
                )
            ):

                return False


            if str(
                d[
                    "lf_solver_tag"
                ]
            ) != LF_SOLVER_TAG:

                return False


            if str(
                d[
                    "hf_solver_tag"
                ]
            ) != HF_SOLVER_TAG:

                return False


            if int(
                d[
                    "op_idx"
                ]
            ) != op_idx:

                return False


            if abs(
                float(
                    d[
                        "beta"
                    ]
                )
                -
                beta_value
            ) > 1.0e-14:

                return False


            if int(
                d[
                    "traj_seed"
                ]
            ) != int(
                operator_traj_seed[
                    op_idx
                ]
            ):

                return False


            if not np.allclose(
                d[
                    "times"
                ],
                times,
                rtol=0.0,
                atol=1.0e-14,
            ):

                return False


            shape_checks = {

                "u0_lf":
                    (
                        N_TRAJ,
                        CFG.nx_lf,
                    ),

                "u0_hf":
                    (
                        N_TRAJ,
                        CFG.nx_hf,
                    ),

                "u0":
                    (
                        N_TRAJ,
                        CFG.nr,
                    ),

                "uc_raw":
                    (
                        N_TRAJ,
                        N_TIME,
                        CFG.nx_lf,
                    ),

                "uf_raw":
                    (
                        N_TRAJ,
                        N_TIME,
                        CFG.nx_hf,
                    ),

                "uc":
                    (
                        N_TRAJ,
                        N_TIME,
                        CFG.nr,
                    ),

                "uf":
                    (
                        N_TRAJ,
                        N_TIME,
                        CFG.nr,
                    ),

                "baseline_relL2":
                    (
                        N_TRAJ,
                        N_TIME,
                    ),

                "baseline_corr":
                    (
                        N_TRAJ,
                        N_TIME,
                    ),

                "lf_mean_drift":
                    (
                        N_TRAJ,
                        N_TIME,
                    ),

                "hf_mean_drift":
                    (
                        N_TRAJ,
                        N_TIME,
                    ),
            }


            for key, shape in shape_checks.items():

                if (
                    d[
                        key
                    ].shape
                    !=
                    shape
                ):

                    return False


            return True


    except Exception:

        return False


# ============================================================
# 15. GENERATE ONE OPERATOR
# ============================================================

def generate_one_operator(
    op_idx: int,
    beta_value: float,
    traj_seed: int,
) -> dict:


    # --------------------------------------------------------
    # 1. Draw GRF bank for this beta/operator
    # --------------------------------------------------------

    f_bank = sample_f_bank(
        N_TRAJ,
        traj_seed,
    )


    # --------------------------------------------------------
    # 2. Evaluate canonical continuous IC on HF256
    # --------------------------------------------------------

    u0_hf_continuous = eval_f_bank_at_x(
        f_bank,
        x_hf,
    )


    # --------------------------------------------------------
    # 3. Apply canonical even-grid HF projection
    # --------------------------------------------------------

    u0_hf = canonicalize_hf_initial_state(
        u0_hf_continuous
    )


    # --------------------------------------------------------
    # 4. SAME canonical IC -> LF72
    # --------------------------------------------------------

    u0_lf = fourier_resample_real(
        u0_hf,
        CFG.nx_lf,
    )


    # --------------------------------------------------------
    # 5. SAME canonical IC -> common100
    # --------------------------------------------------------

    u0_common = fourier_resample_real(
        u0_hf,
        CFG.nr,
    )


    # --------------------------------------------------------
    # 6. Solve LF trajectory
    # --------------------------------------------------------

    uc_raw = solve_lf_fd2_trajectory_batch(
        u0_lf,
        beta_value,
    )


    # --------------------------------------------------------
    # 7. Solve HF trajectory
    # --------------------------------------------------------

    uf_raw = solve_hf_spectral_trajectory_batch(
        u0_hf,
        beta_value,
    )


    # --------------------------------------------------------
    # 8. Model-facing common-grid representations
    # --------------------------------------------------------

    uc = fourier_resample_real(
        uc_raw,
        CFG.nr,
    )


    uf = fourier_resample_real(
        uf_raw,
        CFG.nr,
    )


    # --------------------------------------------------------
    # 9. Baseline metrics
    # --------------------------------------------------------

    baseline_relL2 = row_rel_l2(
        uc,
        uf,
    )


    baseline_corr = row_corr(
        uc,
        uf,
    )


    # --------------------------------------------------------
    # 10. LF spatial-mean conservation
    # --------------------------------------------------------

    # Compare conservation against the true t=0 LF initial state.
    # The first saved snapshot is t=2.505, so uc_raw[:,0] is NOT the IC.
    lf_initial_mean = np.mean(
        u0_lf,
        axis=-1,
        keepdims=True,
    )


    lf_mean = np.mean(
        uc_raw,
        axis=-1,
    )


    lf_mean_drift = (
        lf_mean
        -
        lf_initial_mean
    )


    # --------------------------------------------------------
    # 11. HF spatial-mean conservation
    # --------------------------------------------------------

    # Compare conservation against the true t=0 HF initial state.
    # The first saved snapshot is t=2.505, so uf_raw[:,0] is NOT the IC.
    hf_initial_mean = np.mean(
        u0_hf,
        axis=-1,
        keepdims=True,
    )


    hf_mean = np.mean(
        uf_raw,
        axis=-1,
    )


    hf_mean_drift = (
        hf_mean
        -
        hf_initial_mean
    )


    return {

        "lf_solver_tag":
            np.asarray(
                LF_SOLVER_TAG
            ),

        "hf_solver_tag":
            np.asarray(
                HF_SOLVER_TAG
            ),


        "op_idx":
            np.asarray(
                op_idx,
                dtype=np.int64,
            ),

        "beta":
            np.asarray(
                beta_value,
                dtype=np.float64,
            ),

        "traj_seed":
            np.asarray(
                traj_seed,
                dtype=np.int64,
            ),

        "times":
            times.astype(
                np.float64
            ),


        "u0_lf":
            u0_lf.astype(
                np.float32
            ),

        "u0_hf":
            u0_hf.astype(
                np.float32
            ),

        "u0":
            u0_common.astype(
                np.float32
            ),


        "uc_raw":
            uc_raw.astype(
                np.float32
            ),

        "uf_raw":
            uf_raw.astype(
                np.float32
            ),


        "uc":
            uc.astype(
                np.float32
            ),

        "uf":
            uf.astype(
                np.float32
            ),


        "baseline_relL2":
            baseline_relL2.astype(
                np.float32
            ),

        "baseline_corr":
            baseline_corr.astype(
                np.float32
            ),


        "lf_mean_drift":
            lf_mean_drift.astype(
                np.float64
            ),

        "hf_mean_drift":
            hf_mean_drift.astype(
                np.float64
            ),
    }


# ============================================================
# 16. GENERATE / RESUME ALL OPERATORS
# ============================================================

print(
    "\n" + "=" * 130
)


print(
    "GENERATING LF72 + HF256 KS TRAJECTORY DATASET"
)


print(
    "=" * 130
)


all_start = time.time()


for op_idx in range(
    N_OP
):

    beta_value = float(
        beta[
            op_idx
        ]
    )


    traj_seed = int(
        operator_traj_seed[
            op_idx
        ]
    )


    cache_path = operator_cache_path(
        op_idx
    )


    # --------------------------------------------------------
    # Reuse valid cache
    # --------------------------------------------------------

    if (
        not OVERWRITE_OPERATOR_CACHE
        and
        cache_is_valid(
            cache_path,
            op_idx,
            beta_value,
        )
    ):

        with np.load(
            cache_path,
            allow_pickle=False,
        ) as d:

            rel_final = np.asarray(
                d[
                    "baseline_relL2"
                ][
                    :,
                    -1
                ],
                dtype=np.float64,
            )


            corr_final = np.asarray(
                d[
                    "baseline_corr"
                ][
                    :,
                    -1
                ],
                dtype=np.float64,
            )


            lf_drift = float(
                np.max(
                    np.abs(
                        d[
                            "lf_mean_drift"
                        ]
                    )
                )
            )


        print(
            f"[{op_idx+1:03d}/{N_OP:03d}] "
            f"REUSE CACHE | "
            f"beta={beta_value:.8f} | "
            f"T=3.25 mean={100*np.mean(rel_final):7.3f}% | "
            f"median={100*np.median(rel_final):7.3f}% | "
            f"p90={100*np.percentile(rel_final,90):7.3f}% | "
            f"corr={np.mean(corr_final):.5f} | "
            f"LF-drift={lf_drift:.2e}"
        )


        continue


    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    tic = time.time()


    pack = generate_one_operator(
        op_idx,
        beta_value,
        traj_seed,
    )


    np.savez_compressed(
        cache_path,
        **pack,
    )


    rel_final = np.asarray(
        pack[
            "baseline_relL2"
        ][
            :,
            -1
        ],
        dtype=np.float64,
    )


    corr_final = np.asarray(
        pack[
            "baseline_corr"
        ][
            :,
            -1
        ],
        dtype=np.float64,
    )


    lf_drift = float(
        np.max(
            np.abs(
                pack[
                    "lf_mean_drift"
                ]
            )
        )
    )


    hf_drift = float(
        np.max(
            np.abs(
                pack[
                    "hf_mean_drift"
                ]
            )
        )
    )


    elapsed = (
        time.time()
        -
        tic
    )


    print(
        f"[{op_idx+1:03d}/{N_OP:03d}] "
        f"GENERATED | "
        f"beta={beta_value:.8f} | "
        f"T=3.25 mean={100*np.mean(rel_final):7.3f}% | "
        f"median={100*np.median(rel_final):7.3f}% | "
        f"p90={100*np.percentile(rel_final,90):7.3f}% | "
        f"corr={np.mean(corr_final):.5f} | "
        f"LF-drift={lf_drift:.2e} | "
        f"HF-drift={hf_drift:.2e} | "
        f"{elapsed:.2f}s"
    )


# ============================================================
# 17. ASSEMBLE FULL DATASET
# ============================================================

print(
    "\n" + "=" * 130
)


print(
    "ASSEMBLING FULL DATASET"
)


print(
    "=" * 130
)


u0_lf_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        CFG.nx_lf,
    ),
    dtype=np.float32,
)


u0_hf_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        CFG.nx_hf,
    ),
    dtype=np.float32,
)


u0_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        CFG.nr,
    ),
    dtype=np.float32,
)


uc_raw_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
        CFG.nx_lf,
    ),
    dtype=np.float32,
)


uf_raw_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
        CFG.nx_hf,
    ),
    dtype=np.float32,
)


uc_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
        CFG.nr,
    ),
    dtype=np.float32,
)


uf_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
        CFG.nr,
    ),
    dtype=np.float32,
)


baseline_relL2 = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
    ),
    dtype=np.float32,
)


baseline_corr = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
    ),
    dtype=np.float32,
)


lf_mean_drift_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
    ),
    dtype=np.float64,
)


hf_mean_drift_all = np.empty(
    (
        N_OP,
        N_TRAJ,
        N_TIME,
    ),
    dtype=np.float64,
)


for op_idx in range(
    N_OP
):

    path = operator_cache_path(
        op_idx
    )


    if not cache_is_valid(
        path,
        op_idx,
        float(
            beta[
                op_idx
            ]
        ),
    ):

        raise RuntimeError(
            "Missing or invalid cache for "
            f"operator {op_idx}: {path}"
        )


    with np.load(
        path,
        allow_pickle=False,
    ) as d:


        u0_lf_all[
            op_idx
        ] = d[
            "u0_lf"
        ]


        u0_hf_all[
            op_idx
        ] = d[
            "u0_hf"
        ]


        u0_all[
            op_idx
        ] = d[
            "u0"
        ]


        uc_raw_all[
            op_idx
        ] = d[
            "uc_raw"
        ]


        uf_raw_all[
            op_idx
        ] = d[
            "uf_raw"
        ]


        uc_all[
            op_idx
        ] = d[
            "uc"
        ]


        uf_all[
            op_idx
        ] = d[
            "uf"
        ]


        baseline_relL2[
            op_idx
        ] = d[
            "baseline_relL2"
        ]


        baseline_corr[
            op_idx
        ] = d[
            "baseline_corr"
        ]


        lf_mean_drift_all[
            op_idx
        ] = d[
            "lf_mean_drift"
        ]


        hf_mean_drift_all[
            op_idx
        ] = d[
            "hf_mean_drift"
        ]


# ============================================================
# 18. DATASET DIAGNOSTICS
# ============================================================

train_set = set(
    train_op_idx.tolist()
)


# ------------------------------------------------------------
# Final time
# ------------------------------------------------------------

final_rel_all = baseline_relL2[
    :,
    :,
    -1,
].reshape(
    -1
)


final_rel_train = baseline_relL2[
    train_op_idx,
    :,
    -1,
].reshape(
    -1
)


final_rel_test = baseline_relL2[
    test_op_idx,
    :,
    -1,
].reshape(
    -1
)


final_corr_all = baseline_corr[
    :,
    :,
    -1,
].reshape(
    -1
)


final_corr_train = baseline_corr[
    train_op_idx,
    :,
    -1,
].reshape(
    -1
)


final_corr_test = baseline_corr[
    test_op_idx,
    :,
    -1,
].reshape(
    -1
)


# ------------------------------------------------------------
# All non-zero time states
# ------------------------------------------------------------

# All 150 saved times are nonzero (first saved time is t=2.505).
nonzero_rel_all = baseline_relL2.reshape(
    -1
)


nonzero_rel_train = baseline_relL2[
    train_op_idx
].reshape(
    -1
)


nonzero_rel_test = baseline_relL2[
    test_op_idx
].reshape(
    -1
)


# ------------------------------------------------------------
# Base-time pool
# ------------------------------------------------------------

base_rel_all = baseline_relL2[
    :,
    :,
    base_time_idx,
].reshape(
    -1
)


base_rel_train = baseline_relL2[
    train_op_idx
][
    :,
    :,
    base_time_idx,
].reshape(
    -1
)


base_rel_test = baseline_relL2[
    test_op_idx
][
    :,
    :,
    base_time_idx,
].reshape(
    -1
)


base_corr_all = baseline_corr[
    :,
    :,
    base_time_idx,
].reshape(
    -1
)


base_corr_train = baseline_corr[
    train_op_idx
][
    :,
    :,
    base_time_idx,
].reshape(
    -1
)


base_corr_test = baseline_corr[
    test_op_idx
][
    :,
    :,
    base_time_idx,
].reshape(
    -1
)


# ------------------------------------------------------------
# Near-future OOD pool
# ------------------------------------------------------------

ood_rel_all = baseline_relL2[
    :,
    :,
    ood_time_idx,
].reshape(
    -1
)


ood_rel_train = baseline_relL2[
    train_op_idx
][
    :,
    :,
    ood_time_idx,
].reshape(
    -1
)


ood_rel_test = baseline_relL2[
    test_op_idx
][
    :,
    :,
    ood_time_idx,
].reshape(
    -1
)


ood_corr_all = baseline_corr[
    :,
    :,
    ood_time_idx,
].reshape(
    -1
)


ood_corr_train = baseline_corr[
    train_op_idx
][
    :,
    :,
    ood_time_idx,
].reshape(
    -1
)


ood_corr_test = baseline_corr[
    test_op_idx
][
    :,
    :,
    ood_time_idx,
].reshape(
    -1
)


# ------------------------------------------------------------
# Conservation
# ------------------------------------------------------------

max_lf_mean_drift = float(
    np.max(
        np.abs(
            lf_mean_drift_all
        )
    )
)


max_hf_mean_drift = float(
    np.max(
        np.abs(
            hf_mean_drift_all
        )
    )
)


mean_lf_traj_max_drift = float(
    np.mean(
        np.max(
            np.abs(
                lf_mean_drift_all
            ),
            axis=-1,
        )
    )
)


mean_hf_traj_max_drift = float(
    np.mean(
        np.max(
            np.abs(
                hf_mean_drift_all
            ),
            axis=-1,
        )
    )
)


summary = {

    "final_time_RelL2_percent": {

        "overall":
            describe_percent(
                final_rel_all
            ),

        "train":
            describe_percent(
                final_rel_train
            ),

        "test":
            describe_percent(
                final_rel_test
            ),
    },


    "all_nonzero_times_RelL2_percent": {

        "overall":
            describe_percent(
                nonzero_rel_all
            ),

        "train":
            describe_percent(
                nonzero_rel_train
            ),

        "test":
            describe_percent(
                nonzero_rel_test
            ),
    },


    "base_time_RelL2_percent": {

        "time_range":
            [
                CFG.base_t_min,
                CFG.base_t_max,
            ],

        "n_time_slots":
            int(
                len(
                    base_time_idx
                )
            ),

        "overall":
            describe_percent(
                base_rel_all
            ),

        "train":
            describe_percent(
                base_rel_train
            ),

        "test":
            describe_percent(
                base_rel_test
            ),
    },


    "base_time_correlation": {

        "overall_mean":
            float(
                np.mean(
                    base_corr_all
                )
            ),

        "train_mean":
            float(
                np.mean(
                    base_corr_train
                )
            ),

        "test_mean":
            float(
                np.mean(
                    base_corr_test
                )
            ),

        "overall_median":
            float(
                np.median(
                    base_corr_all
                )
            ),

        "overall_p10":
            float(
                np.percentile(
                    base_corr_all,
                    10,
                )
            ),

        "overall_p05":
            float(
                np.percentile(
                    base_corr_all,
                    5,
                )
            ),

        "overall_min":
            float(
                np.min(
                    base_corr_all
                )
            ),
    },


    "ood_time_RelL2_percent": {

        "time_range":
            [
                float(
                    ood_times[0]
                ),
                float(
                    ood_times[-1]
                ),
            ],

        "n_time_slots":
            int(
                len(
                    ood_time_idx
                )
            ),

        "overall":
            describe_percent(
                ood_rel_all
            ),

        "train":
            describe_percent(
                ood_rel_train
            ),

        "test":
            describe_percent(
                ood_rel_test
            ),
    },


    "ood_time_correlation": {

        "overall_mean":
            float(
                np.mean(
                    ood_corr_all
                )
            ),

        "train_mean":
            float(
                np.mean(
                    ood_corr_train
                )
            ),

        "test_mean":
            float(
                np.mean(
                    ood_corr_test
                )
            ),

        "overall_median":
            float(
                np.median(
                    ood_corr_all
                )
            ),

        "overall_p10":
            float(
                np.percentile(
                    ood_corr_all,
                    10,
                )
            ),

        "overall_p05":
            float(
                np.percentile(
                    ood_corr_all,
                    5,
                )
            ),

        "overall_min":
            float(
                np.min(
                    ood_corr_all
                )
            ),
    },


    "final_time_correlation": {

        "overall_mean":
            float(
                np.mean(
                    final_corr_all
                )
            ),

        "train_mean":
            float(
                np.mean(
                    final_corr_train
                )
            ),

        "test_mean":
            float(
                np.mean(
                    final_corr_test
                )
            ),

        "overall_median":
            float(
                np.median(
                    final_corr_all
                )
            ),

        "overall_min":
            float(
                np.min(
                    final_corr_all
                )
            ),
    },


    "mean_conservation": {

        "LF_max_abs_mean_drift":
            max_lf_mean_drift,

        "HF_max_abs_mean_drift":
            max_hf_mean_drift,

        "LF_mean_trajectory_max_abs_mean_drift":
            mean_lf_traj_max_drift,

        "HF_mean_trajectory_max_abs_mean_drift":
            mean_hf_traj_max_drift,
    },
}


# ============================================================
# 19. TIME-WISE BASELINE TABLE
# ============================================================

time_rows = []


for ti, t_value in enumerate(
    times
):

    r_all = baseline_relL2[
        :,
        :,
        ti,
    ].reshape(
        -1
    ).astype(
        np.float64
    )


    r_train = baseline_relL2[
        train_op_idx,
        :,
        ti,
    ].reshape(
        -1
    ).astype(
        np.float64
    )


    r_test = baseline_relL2[
        test_op_idx,
        :,
        ti,
    ].reshape(
        -1
    ).astype(
        np.float64
    )


    c_all = baseline_corr[
        :,
        :,
        ti,
    ].reshape(
        -1
    ).astype(
        np.float64
    )


    c_train = baseline_corr[
        train_op_idx,
        :,
        ti,
    ].reshape(
        -1
    ).astype(
        np.float64
    )


    c_test = baseline_corr[
        test_op_idx,
        :,
        ti,
    ].reshape(
        -1
    ).astype(
        np.float64
    )


    time_rows.append({

        "time_idx":
            int(
                ti
            ),

        "time":
            float(
                t_value
            ),


        "overall_mean_RelL2_pct":
            float(
                100.0
                *
                np.mean(
                    r_all
                )
            ),

        "overall_median_RelL2_pct":
            float(
                100.0
                *
                np.median(
                    r_all
                )
            ),

        "overall_p90_RelL2_pct":
            float(
                100.0
                *
                np.percentile(
                    r_all,
                    90,
                )
            ),

        "overall_p95_RelL2_pct":
            float(
                100.0
                *
                np.percentile(
                    r_all,
                    95,
                )
            ),


        "train_mean_RelL2_pct":
            float(
                100.0
                *
                np.mean(
                    r_train
                )
            ),

        "train_median_RelL2_pct":
            float(
                100.0
                *
                np.median(
                    r_train
                )
            ),

        "train_p90_RelL2_pct":
            float(
                100.0
                *
                np.percentile(
                    r_train,
                    90,
                )
            ),


        "test_mean_RelL2_pct":
            float(
                100.0
                *
                np.mean(
                    r_test
                )
            ),

        "test_median_RelL2_pct":
            float(
                100.0
                *
                np.median(
                    r_test
                )
            ),

        "test_p90_RelL2_pct":
            float(
                100.0
                *
                np.percentile(
                    r_test,
                    90,
                )
            ),

        "test_p95_RelL2_pct":
            float(
                100.0
                *
                np.percentile(
                    r_test,
                    95,
                )
            ),


        "overall_mean_corr":
            float(
                np.mean(
                    c_all
                )
            ),

        "train_mean_corr":
            float(
                np.mean(
                    c_train
                )
            ),

        "test_mean_corr":
            float(
                np.mean(
                    c_test
                )
            ),
    })


time_df = pd.DataFrame(
    time_rows
)


time_csv = (
    OUTPUT_ROOT
    /
    "baseline_by_time.csv"
)


time_df.to_csv(
    time_csv,
    index=False,
)


# ============================================================
# 20. OPERATOR-WISE FINAL-TIME TABLE
# ============================================================

op_rows = []


for op_idx in range(
    N_OP
):

    r = baseline_relL2[
        op_idx,
        :,
        -1,
    ].astype(
        np.float64
    )


    c = baseline_corr[
        op_idx,
        :,
        -1,
    ].astype(
        np.float64
    )


    op_rows.append({

        "op_idx":
            int(
                op_idx
            ),

        "beta":
            float(
                beta[
                    op_idx
                ]
            ),

        "split":
            (
                "train"
                if
                op_idx
                in
                train_set
                else
                "test"
            ),


        "mean_RelL2_pct_Tfinal":
            float(
                100.0
                *
                np.mean(
                    r
                )
            ),

        "std_RelL2_pct_Tfinal":
            float(
                100.0
                *
                np.std(
                    r
                )
            ),

        "median_RelL2_pct_Tfinal":
            float(
                100.0
                *
                np.median(
                    r
                )
            ),

        "p90_RelL2_pct_Tfinal":
            float(
                100.0
                *
                np.percentile(
                    r,
                    90,
                )
            ),

        "p95_RelL2_pct_Tfinal":
            float(
                100.0
                *
                np.percentile(
                    r,
                    95,
                )
            ),

        "max_RelL2_pct_Tfinal":
            float(
                100.0
                *
                np.max(
                    r
                )
            ),


        "mean_corr_Tfinal":
            float(
                np.mean(
                    c
                )
            ),

        "min_corr_Tfinal":
            float(
                np.min(
                    c
                )
            ),
    })


op_df = pd.DataFrame(
    op_rows
)


op_csv = (
    OUTPUT_ROOT
    /
    "baseline_by_operator_Tfinal.csv"
)


op_df.to_csv(
    op_csv,
    index=False,
)


# ============================================================
# 21. METADATA
# ============================================================

metadata = {

    "dataset_name":
        dataset_stem,


    "generation_scope": (
        "generated from the KS PDE as a complete dense "
        "multi-fidelity trajectory dataset; demonstrations "
        "and query times are sampled later by the training code"
    ),


    "pde":
        "u_t + u*u_x + u_xx + beta*u_xxxx = 0",


    "domain":
        [
            0.0,
            CFG.L,
        ],


    "periodic_bc":
        True,


    "operator_parameter":
        "beta",

    "operator_definition":
        "beta",

    "trajectory_identity":
        "fixed (beta,u0)",


    "beta_sampling": (
        f"{N_OP} uniformly spaced values including endpoints "
        f"[{CFG.beta_min},{CFG.beta_max}]"
    ),


    "beta_min":
        CFG.beta_min,


    "beta_max":
        CFG.beta_max,


    "n_operator":
        N_OP,


    "n_trajectory_per_operator":
        N_TRAJ,


    "n_train_operator":
        N_TRAIN,


    "n_test_operator":
        N_TEST,

    "n_group_A_operator":
        int(len(group_a_op_idx)),

    "n_group_B_operator":
        int(len(group_b_op_idx)),

    "group_a_op_idx":
        group_a_op_idx.tolist(),

    "group_b_op_idx":
        group_b_op_idx.tolist(),


    "operator_split_seed":
        CFG.split_seed,


    "split_rule": (
        "fixed seed-42 100/100 split by beta/operator; Group B is the "
        "first 100 entries of rng.permutation(200), Group A is the remainder"
    ),


    # --------------------------------------------------------
    # Spatial grids
    # --------------------------------------------------------

    "nx_lf":
        CFG.nx_lf,


    "nx_hf":
        CFG.nx_hf,


    "nr_common":
        CFG.nr,


    # --------------------------------------------------------
    # Time
    # --------------------------------------------------------

    "t_final":
        CFG.t_final,


    "solver_dt":
        CFG.dt,


    "base_snapshot_dt":
        CFG.base_snapshot_dt,

    "ood_snapshot_dt":
        CFG.ood_snapshot_dt,


    "n_solver_steps":
        n_steps,


    "n_saved_times":
        N_TIME,


    "saved_times":
        times.tolist(),


    "base_time_min":
        CFG.base_t_min,


    "base_time_max":
        CFG.base_t_max,


    "base_time_idx":
        base_time_idx.tolist(),


    "base_times":
        base_times.tolist(),


    "ood_time_min":
        CFG.ood_t_min,


    "ood_time_max":
        CFG.ood_t_max,


    "ood_time_idx":
        ood_time_idx.tolist(),


    "ood_times":
        ood_times.tolist(),


    "mixed_time_idx":
        mixed_time_idx.tolist(),


    "mixed_times":
        mixed_times.tolist(),


    "time_pool_note": (
        "Exact uniform saved-time design: 100 BASE times 2.505:0.005:3.000 "
        "plus 50 OOD times 3.005:0.005:3.250. No temporal interpolation. "
        "Later ICL training samples M+1 distinct time tokens uniformly without replacement."
    ),

    "natural_group_A_OOD_fraction": NATURAL_GROUP_A_OOD_FRACTION,


    # --------------------------------------------------------
    # LF numerical method
    # --------------------------------------------------------

    "lf_spatial_solver":
        "second-order centered finite difference",


    "lf_nonlinear_discretization": (
        "conservative centered derivative of f(u)=u^2/2: "
        "-[f(u_{i+1})-f(u_{i-1})]/(2 dx)"
    ),


    "lf_second_derivative": (
        "(u_{i+1}-2u_i+u_{i-1})/dx^2"
    ),


    "lf_fourth_derivative":
        "D4 = D2(D2)",


    "lf_nominal_spatial_order":
        2,


    "lf_time_integrator":
        "ETDRK4",


    "lf_solver_tag":
        LF_SOLVER_TAG,


    # --------------------------------------------------------
    # HF numerical method
    # --------------------------------------------------------

    "hf_spatial_solver":
        "periodic Fourier pseudospectral",


    "hf_quadratic_nonlinearity":
        "-0.5*d_x(u^2)",


    "hf_dealiasing":
        "3/2-rule zero padding",


    "hf_time_integrator":
        "ETDRK4",


    "hf_solver_tag":
        HF_SOLVER_TAG,


    # --------------------------------------------------------
    # IC
    # --------------------------------------------------------

    "initial_condition":
        "u0(x)=f(sin^2(x/2))",


    "grf_mean_function":
        0.0,


    "grf_covariance":
        "sigma^2 exp(-(z-z')^2/(2 ell^2))",


    "grf_lengthscale":
        CFG.grf_lengthscale,


    "grf_sigma":
        CFG.grf_sigma,


    "grf_z_grid_n":
        CFG.grf_z_grid_n,


    "grf_jitter":
        CFG.grf_jitter,


    "ic_construction": (
        "sample f(z), evaluate on HF256, apply the HF even-grid "
        "Nyquist convention to define the canonical numerical IC, "
        "then Fourier-resample the same canonical IC to LF72 "
        "and common100"
    ),


    "ic_base_seed":
        CFG.ic_base_seed,


    "operator_seed_stride":
        CFG.operator_seed_stride,


    "operator_seed_rule": (
        "traj_seed(op_idx) = "
        "ic_base_seed + op_idx*operator_seed_stride"
    ),


    "independent_ic_bank_per_operator":
        True,


    # --------------------------------------------------------
    # Representation / dtype
    # --------------------------------------------------------

    "common_grid_mapping":
        "periodic Fourier resampling using scipy.signal.resample",


    "compute_dtype":
        "float64/complex128",


    "save_dtype": (
        "float32 for state fields/errors; "
        "float64 for coordinates/beta/time/mean-drift; "
        "int64 for indices/seeds"
    ),


    # --------------------------------------------------------
    # ICL task
    # --------------------------------------------------------

    "dataset_does_not_fix_prompt_sampling":
        True,


    "intended_query_t_formulation": (
        "within one fixed (beta,u0) trajectory, later training "
        "samples M temporal demonstrations (t,uc(t),uf(t)) and "
        "a distinct query (t_q,uc(t_q)) to predict uf(t_q); "
        "no-mixed Group A/B use BASE 2.505:0.005:3.000; mixed Group A uses "
        "100 BASE + 50 OOD at 3.005:0.005:3.250; mixed Group B remains BASE-only"
    ),


    "prediction_target_note":
        "direct uf(t_q), not residual learning",


    "baseline_definition": (
        "pointwise-in-time common-grid RelL2/correlation "
        "between FD2-LF72 and spectral-HF256"
    ),


    "baseline_summary":
        summary,


    "config":
        asdict(
            CFG
        ),
}


metadata_path = (
    OUTPUT_ROOT
    /
    "metadata.json"
)


with open(
    metadata_path,
    "w",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )


# ============================================================
# 22. SAVE MASTER NPZ
# ============================================================

master_path = (
    OUTPUT_ROOT
    /
    f"{dataset_stem}.npz"
)


save_payload = dict(

    beta=
        beta.astype(
            np.float64
        ),

    operator_definition=
        np.asarray(
            "beta"
        ),

    times=
        times.astype(
            np.float64
        ),

    snapshot_steps=
        snapshot_steps.astype(
            np.int64
        ),

    base_time_mask=
        base_time_mask.astype(
            np.bool_
        ),

    base_time_idx=
        base_time_idx.astype(
            np.int64
        ),

    base_times=
        base_times.astype(
            np.float64
        ),


    ood_time_mask=
        ood_time_mask.astype(
            np.bool_
        ),

    ood_time_idx=
        ood_time_idx.astype(
            np.int64
        ),

    ood_times=
        ood_times.astype(
            np.float64
        ),


    mixed_time_mask=
        mixed_time_mask.astype(
            np.bool_
        ),

    mixed_time_idx=
        mixed_time_idx.astype(
            np.int64
        ),

    mixed_times=
        mixed_times.astype(
            np.float64
        ),


    x_lf=
        x_lf.astype(
            np.float64
        ),

    x_hf=
        x_hf.astype(
            np.float64
        ),

    x=
        x_common.astype(
            np.float64
        ),


    u0_lf=
        u0_lf_all,

    u0_hf=
        u0_hf_all,

    u0=
        u0_all,


    uc_raw=
        uc_raw_all,

    uf_raw=
        uf_raw_all,


    uc=
        uc_all,

    uf=
        uf_all,


    baseline_relL2=
        baseline_relL2,

    baseline_corr=
        baseline_corr,


    lf_mean_drift=
        lf_mean_drift_all,

    hf_mean_drift=
        hf_mean_drift_all,


    operator_traj_seed=
        operator_traj_seed,

    train_op_idx=
        train_op_idx,

    test_op_idx=
        test_op_idx,

    group_a_op_idx=
        group_a_op_idx,

    group_b_op_idx=
        group_b_op_idx,


    T=
        np.asarray(
            CFG.t_final,
            dtype=np.float64,
        ),

    dt=
        np.asarray(
            CFG.dt,
            dtype=np.float64,
        ),

    base_snapshot_dt=
        np.asarray(CFG.base_snapshot_dt, dtype=np.float64),

    ood_snapshot_dt=
        np.asarray(CFG.ood_snapshot_dt, dtype=np.float64),


    nx_lf=
        np.asarray(
            CFG.nx_lf,
            dtype=np.int64,
        ),

    nx_hf=
        np.asarray(
            CFG.nx_hf,
            dtype=np.int64,
        ),

    nr=
        np.asarray(
            CFG.nr,
            dtype=np.int64,
        ),

    n_traj=
        np.asarray(
            N_TRAJ,
            dtype=np.int64,
        ),

    n_time=
        np.asarray(
            N_TIME,
            dtype=np.int64,
        ),


    lf_solver_tag=
        np.asarray(
            LF_SOLVER_TAG
        ),

    hf_solver_tag=
        np.asarray(
            HF_SOLVER_TAG
        ),


    metadata_json=
        np.asarray(
            json.dumps(
                metadata
            )
        ),
)


print(
    "\nSaving master dataset:"
)


print(
    master_path
)


if SAVE_COMPRESSED:

    np.savez_compressed(
        master_path,
        **save_payload,
    )

else:

    np.savez(
        master_path,
        **save_payload,
    )


# ============================================================
# 23. SHA256
# ============================================================

def sha256_file(
    path: Path,
) -> str:


    h = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024
                    *
                    1024
                ),
            b"",
        ):

            h.update(
                chunk
            )


    return h.hexdigest()


sha256 = sha256_file(
    master_path
)


sha_path = (
    OUTPUT_ROOT
    /
    "sha256.txt"
)


sha_path.write_text(
    f"{sha256}  {master_path.name}\n"
)


# ============================================================
# 24. BASELINE SUMMARY JSON
# ============================================================

summary_payload = {

    "dataset_path":
        str(
            master_path
        ),


    "sha256":
        sha256,


    "n_operator":
        N_OP,


    "n_trajectory_per_operator":
        N_TRAJ,


    "n_saved_times":
        N_TIME,


    "saved_time_range":
        [
            float(
                times[
                    0
                ]
            ),
            float(
                times[
                    -1
                ]
            ),
        ],


    "base_snapshot_dt":
        CFG.base_snapshot_dt,

    "ood_snapshot_dt":
        CFG.ood_snapshot_dt,


    "beta_range":
        [
            float(
                beta.min()
            ),
            float(
                beta.max()
            ),
        ],


    "nx_lf":
        CFG.nx_lf,


    "nx_hf":
        CFG.nx_hf,


    "nr":
        CFG.nr,


    "t_final":
        CFG.t_final,


    "solver_dt":
        CFG.dt,


    "lf_solver":
        "second-order centered FD + ETDRK4",


    "hf_solver":
        "3/2-dealiased Fourier pseudospectral + ETDRK4",


    "final_time_RelL2_percent":
        summary[
            "final_time_RelL2_percent"
        ],


    "all_nonzero_times_RelL2_percent":
        summary[
            "all_nonzero_times_RelL2_percent"
        ],


    "base_time_range":
        [
            CFG.base_t_min,
            CFG.base_t_max,
        ],


    "base_time_idx":
        base_time_idx.tolist(),


    "base_time_RelL2_percent":
        summary[
            "base_time_RelL2_percent"
        ],


    "base_time_correlation":
        summary[
            "base_time_correlation"
        ],


    "ood_time_range":
        [
            float(
                ood_times[0]
            ),
            float(
                ood_times[-1]
            ),
        ],


    "ood_time_idx":
        ood_time_idx.tolist(),


    "ood_time_RelL2_percent":
        summary[
            "ood_time_RelL2_percent"
        ],


    "ood_time_correlation":
        summary[
            "ood_time_correlation"
        ],


    "final_time_correlation":
        summary[
            "final_time_correlation"
        ],


    "mean_conservation":
        summary[
            "mean_conservation"
        ],
}


summary_json_path = (
    OUTPUT_ROOT
    /
    "baseline_summary.json"
)


with open(
    summary_json_path,
    "w",
) as f:

    json.dump(
        summary_payload,
        f,
        indent=2,
    )


# ============================================================
# 25. PLOTS
# ============================================================

# ------------------------------------------------------------
# Final-time histogram
# ------------------------------------------------------------

plt.figure(
    figsize=(
        7.5,
        5.0,
    )
)


plt.hist(
    100.0
    *
    final_rel_all,
    bins=50,
)


plt.xlabel(
    "FD2-LF72 -> spectral-HF256 RelL2 at T=3.25 (%)"
)


plt.ylabel(
    "Count"
)


plt.title(
    "KS raw multi-fidelity baseline at T=3.25"
)


plt.tight_layout()


plt.savefig(
    OUTPUT_ROOT
    /
    "baseline_T3p25_histogram.png",
    dpi=180,
)


plt.close()


# ------------------------------------------------------------
# Error vs time
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8.0,
        5.2,
    )
)


plt.plot(
    time_df[
        "time"
    ],
    time_df[
        "overall_mean_RelL2_pct"
    ],
    label="mean",
)


plt.plot(
    time_df[
        "time"
    ],
    time_df[
        "overall_median_RelL2_pct"
    ],
    label="median",
)


plt.plot(
    time_df[
        "time"
    ],
    time_df[
        "overall_p90_RelL2_pct"
    ],
    label="p90",
)


plt.xlabel(
    "Time"
)


plt.ylabel(
    "FD2-LF72 -> spectral-HF256 RelL2 (%)"
)


plt.title(
    "KS multi-fidelity discrepancy over time"
)


plt.legend()


plt.tight_layout()


plt.savefig(
    OUTPUT_ROOT
    /
    "baseline_vs_time.png",
    dpi=180,
)


plt.close()


# ------------------------------------------------------------
# Final-time discrepancy vs beta
# ------------------------------------------------------------

plt.figure(
    figsize=(
        7.5,
        5.0,
    )
)


plt.plot(
    op_df[
        "beta"
    ],
    op_df[
        "mean_RelL2_pct_Tfinal"
    ],
    label="operator mean",
)


plt.plot(
    op_df[
        "beta"
    ],
    op_df[
        "median_RelL2_pct_Tfinal"
    ],
    label="operator median",
)


plt.plot(
    op_df[
        "beta"
    ],
    op_df[
        "p90_RelL2_pct_Tfinal"
    ],
    label="operator p90",
)


plt.xlabel(
    r"$\beta$"
)


plt.ylabel(
    "Relative L2 error at T=3.25 (%)"
)


plt.title(
    "KS FD2-LF72 -> spectral-HF256 baseline vs beta"
)


plt.legend()


plt.tight_layout()


plt.savefig(
    OUTPUT_ROOT
    /
    "baseline_T3p25_vs_beta.png",
    dpi=180,
)


plt.close()


# ============================================================
# 26. FINAL AUDIT
# ============================================================

print(
    "\n" + "=" * 130
)


print(
    "DATASET GENERATION COMPLETE"
)


print(
    "=" * 130
)


print(
    "\nShapes:"
)


print(
    "  beta               :",
    beta.shape,
)


print(
    "  times              :",
    times.shape,
)


print(
    "  u0_lf              :",
    u0_lf_all.shape,
)


print(
    "  u0_hf              :",
    u0_hf_all.shape,
)


print(
    "  u0                 :",
    u0_all.shape,
)


print(
    "  uc_raw             :",
    uc_raw_all.shape,
)


print(
    "  uf_raw             :",
    uf_raw_all.shape,
)


print(
    "  uc                 :",
    uc_all.shape,
)


print(
    "  uf                 :",
    uf_all.shape,
)


print(
    "  baseline_relL2     :",
    baseline_relL2.shape,
)


print(
    "  baseline_corr      :",
    baseline_corr.shape,
)


print(
    "  lf_mean_drift      :",
    lf_mean_drift_all.shape,
)


print(
    "  hf_mean_drift      :",
    hf_mean_drift_all.shape,
)


print(
    "  train_op_idx       :",
    train_op_idx.shape,
)


print(
    "  test_op_idx        :",
    test_op_idx.shape,
)


# ============================================================
# FINAL TIME
# ============================================================

print(
    "\nFinal-time T=3.25 raw FD2-LF72 -> spectral-HF256 baseline:"
)


for split_name in (
    "overall",
    "train",
    "test",
):

    s = summary[
        "final_time_RelL2_percent"
    ][
        split_name
    ]


    print(
        f"  {split_name.upper():7s} | "
        f"mean={s['mean']:.4f}% | "
        f"std={s['std']:.4f}% | "
        f"median={s['median']:.4f}% | "
        f"p90={s['p90']:.4f}% | "
        f"p95={s['p95']:.4f}% | "
        f"p99={s['p99']:.4f}% | "
        f"max={s['max']:.4f}%"
    )


# ============================================================
# SELECTED TIMES
# ============================================================

print(
    "\nSelected time-wise baseline checkpoints:"
)


for t_show in (
    2.505, 2.600, 2.700, 2.800, 2.900, 3.000,
    3.050, 3.100, 3.150, 3.200, 3.250,
):
    ti = int(np.argmin(np.abs(times - t_show)))
    if abs(float(times[ti]) - t_show) > 5.0e-12:
        raise RuntimeError(f"Requested report time {t_show} is not saved exactly.")
    row = time_df.iloc[ti]


    print(
        f"  t={t_show:4.2f} | "
        f"mean={row['overall_mean_RelL2_pct']:.4f}% | "
        f"median={row['overall_median_RelL2_pct']:.4f}% | "
        f"p90={row['overall_p90_RelL2_pct']:.4f}% | "
        f"corr={row['overall_mean_corr']:.6f}"
    )


# ============================================================
# BASE-TIME POOL
# ============================================================

print(
    "\nBase-time pool raw "
    "FD2-LF72 -> spectral-HF256 baseline:"
)


print(
    f"  t in "
    f"[{CFG.base_t_min:.3f},{CFG.base_t_max:.3f}] "
    f"({len(base_time_idx)} slots)"
)


for split_name in (
    "overall",
    "train",
    "test",
):

    s = summary[
        "base_time_RelL2_percent"
    ][
        split_name
    ]


    print(
        f"  {split_name.upper():7s} | "
        f"mean={s['mean']:.4f}% | "
        f"std={s['std']:.4f}% | "
        f"median={s['median']:.4f}% | "
        f"p90={s['p90']:.4f}% | "
        f"p95={s['p95']:.4f}% | "
        f"p99={s['p99']:.4f}% | "
        f"max={s['max']:.4f}%"
    )


corr_summary = summary[
    "base_time_correlation"
]


print(
    "\nBase-pool correlation:"
)


print(
    f"  overall mean   = "
    f"{corr_summary['overall_mean']:.6f}"
)


print(
    f"  train mean     = "
    f"{corr_summary['train_mean']:.6f}"
)


print(
    f"  test mean      = "
    f"{corr_summary['test_mean']:.6f}"
)


print(
    f"  overall median = "
    f"{corr_summary['overall_median']:.6f}"
)


print(
    f"  overall p10    = "
    f"{corr_summary['overall_p10']:.6f}"
)


print(
    f"  overall p05    = "
    f"{corr_summary['overall_p05']:.6f}"
)


print(
    f"  overall min    = "
    f"{corr_summary['overall_min']:.6f}"
)



# ============================================================
# NEAR-FUTURE OOD POOL
# ============================================================

print(
    "\nNear-future OOD pool raw "
    "FD2-LF72 -> spectral-HF256 baseline:"
)


print(
    f"  t in "
    f"[{ood_times[0]:.3f},{ood_times[-1]:.3f}] "
    f"({len(ood_time_idx)} slots)"
)


for split_name in (
    "overall",
    "train",
    "test",
):

    s = summary[
        "ood_time_RelL2_percent"
    ][
        split_name
    ]


    print(
        f"  {split_name.upper():7s} | "
        f"mean={s['mean']:.4f}% | "
        f"std={s['std']:.4f}% | "
        f"median={s['median']:.4f}% | "
        f"p90={s['p90']:.4f}% | "
        f"p95={s['p95']:.4f}% | "
        f"p99={s['p99']:.4f}% | "
        f"max={s['max']:.4f}%"
    )


ood_corr_summary = summary[
    "ood_time_correlation"
]


print(
    "\nOOD-pool correlation:"
)


print(
    f"  overall mean   = "
    f"{ood_corr_summary['overall_mean']:.6f}"
)


print(
    f"  train mean     = "
    f"{ood_corr_summary['train_mean']:.6f}"
)


print(
    f"  test mean      = "
    f"{ood_corr_summary['test_mean']:.6f}"
)


print(
    f"  overall median = "
    f"{ood_corr_summary['overall_median']:.6f}"
)


print(
    f"  overall p10    = "
    f"{ood_corr_summary['overall_p10']:.6f}"
)


print(
    f"  overall p05    = "
    f"{ood_corr_summary['overall_p05']:.6f}"
)


print(
    f"  overall min    = "
    f"{ood_corr_summary['overall_min']:.6f}"
)


# ============================================================
# CONSERVATION
# ============================================================

print(
    "\nSpatial-mean conservation:"
)


print(
    f"  LF max abs drift        = "
    f"{max_lf_mean_drift:.6e}"
)


print(
    f"  HF max abs drift        = "
    f"{max_hf_mean_drift:.6e}"
)


print(
    f"  LF mean traj max drift  = "
    f"{mean_lf_traj_max_drift:.6e}"
)


print(
    f"  HF mean traj max drift  = "
    f"{mean_hf_traj_max_drift:.6e}"
)


# ============================================================
# FILES
# ============================================================

print(
    "\nFiles:"
)


print(
    "  master NPZ       :",
    master_path,
)


print(
    "  metadata         :",
    metadata_path,
)


print(
    "  baseline by time :",
    time_csv,
)


print(
    "  baseline by beta :",
    op_csv,
)


print(
    "  summary           :",
    summary_json_path,
)


print(
    "  sha256            :",
    sha_path,
)


print(
    "  plots             :",
    OUTPUT_ROOT
    /
    "baseline_vs_time.png",
)


print(
    "                      ",
    OUTPUT_ROOT
    /
    "baseline_T3p25_histogram.png",
)


print(
    "                      ",
    OUTPUT_ROOT
    /
    "baseline_T3p25_vs_beta.png",
)


print(
    "\nSHA256:"
)


print(
    " ",
    sha256
)


print(
    "\nTotal elapsed: %.1f s"
    %
    (
        time.time()
        -
        all_start
    )
)


print(
    "=" * 130
)

KS MULTI-FIDELITY QUERY-T DATASET GENERATOR
DEVICE              : cuda
QUICK_TEST          : False
OUTPUT_ROOT         : /kaggle/working/ks_mf_queryt_beta070_085_FD2LF72_SpecHF256_T3p25_B100late_OOD50_dt001_Nop200_Ntraj20_Nt150_Nr100
PDE                 : u_t + u*u_x + u_xx + beta*u_xxxx = 0
beta range          : [0.070000, 0.085000]
N operators         : 200
trajectories/op     : 20
train / test beta   : 100 / 100
LF / HF / Nr        : 72 / 256 / 100
LF solver           : 2nd-order centered FD + ETDRK4
HF solver           : 3/2-dealiased Fourier pseudospectral + ETDRK4
T_max / solver dt   : 3.25 / 0.001
BASE snapshot dt    : 0.005
OOD snapshot dt     : 0.005
N snapshots         : 150
saved times         : 2.505 ... 3.250
base pool           : 2.505 ... 3.000 (100 slots)
near-future OOD    : 3.005 ... 3.250 (50 slots)
mixed A pool       : 2.505 ... 3.250 (150 slots)
IC                  : u0(x)=f(sin^2(x/2))
GRF                 : SE kernel, ell=0.5, sigma=1.0
fresh IC base seed  : 20260